# Khai báo thư viện và Tài nguyên

In [ ]:
import pandas as pd
import re
import os
from pyvi import ViTokenizer

# 1. BỘ TỪ ĐIỂN 
teencode_dict = {
    # Nhóm viết tắt thông thường
    'k': 'không', 'ko': 'không', 'kh': 'không', 'hông': 'không', 'kg': 'không', 'khong': 'không',
    'j': 'gì', 'dzì': 'gì', 'zì': 'gì', 'ng': 'người', 'cx': 'cũng', 'bt': 'bình thường', 
    'kb': 'không biết', 'z': 'vậy', 'v': 'vậy', 'zậy': 'vậy', 't': 'tao', 'mk': 'mình', 'm': 'mày',
    'bn': 'bạn', 'mn': 'mọi người', 'đt': 'điện thoại', 'fb': 'facebook', 'ytb': 'youtube', 
    'cmt': 'bình luận', 'mv': 'video', 'tr': 'triệu', 'ms': 'mới', 'nt': 'nhắn tin', 'rep': 'trả lời',
    'ae': 'anh em', 'ad': 'quản trị viên', 'ca': 'công an',
    'đc': 'được', 'chx': 'chưa','gđ': 'gia đình', 'nhìu': 'nhiều', 'mih': 'mình','ts': 'tại sao','tn': 'thế nào','ny': 'người yêu',
    'stt': 'số thứ tự','p': 'phút' ,'ak': 'à', 'ợ': 'ạ', 'uk': 'ừ', 'thg' : 'thằng', 'bth': 'bình thường',

    # Nhóm Độc hại / Tục tĩu 
    'vcl': 'vãi cả lồn', 'vl': 'vãi lồn', 'vkl': 'vãi cả lồn', 'vlon': 'vãi lồn', 'cl': 'cái lồn',
    'vcll': 'vãi cả lồn luôn', 'vc' : 'vãi cặc', 'vđ' : 'vãi đái', 'tđn' : 'thế đéo nào', 'đ' : 'đéo',
    'đ*': 'địt', 'đ**': 'đéo', 'cút' : 'cứt', 'dái' : 'dái', 'hạch' : 'dở như hạch', 'bại não' : 'ngu',
    'mất não': 'ngu đéo chịu được', 'cln': 'cái lồn',
    'lz': 'lồn', 'lol': 'lồn', 'ln': 'lồn', 'cc': 'cặc', 'c-c': 'cặc', 'c*c': 'cặc',
    'db': 'đầu buồi', 'đb': 'đầu buồi', 'tml': 'thằng mặt lồn', 'ccmm': 'con cặc mẹ mày',
    'đ': 'đéo', 'đ\'': 'đéo', 'đeo': 'đéo', 'xl': 'xạo lồn', 'xlon': 'xạo lồn', 'bl': 'bú lồn',
    'cailon': 'cái lồn', 'phổng đạn': 'phản động', '3/': 'ba que', 'parky': 'bắc kỳ',
    'sủa': 'nói càn', 'óc': 'óc chó', 'lồ-' : 'lồn', 'iar' : 'ỉa', 'dcm' : 'địt con mẹ', 'dit' : 'địt',
    'mọe': 'mẹ', 'đouma' : 'đụ má', 'lôn' : 'lồn', 'thử đầm' : 'thủ dâm', 'cặt' : 'cặc', 

    #Nhóm tiếng anh 
    'beef': 'tranh chấp', 'bưng bô': 'nịnh bợ', 're' : 'rên', 'under': 'underground', 'fame': 'sự nổi tiếng',
     'đpq': 'Đỗ Phú Quý', 'qk': 'Quốc Kang', 'st': 'Sơn Tùng', 'tđ': 'Trấn Thành', 'atsh': 'Anh Trai Say Hi', 
    # Nhóm Nói lái 
    'hit như đạt': 'hát như địt', 'hốt như làn': 'hát như lồn', 'nhứk đít': 'nhức đít',
    'chuông kì bu': 'chu kỳ buồi', 'gọi bằng mồm': 'gom bằng rác', 'hồn như lát' : 'hát như lồn'
}

# 2. Tải danh sách Stopwords
try:
    with open('data/vietnamese-stopwords.txt', 'r', encoding='utf-8') as f:
        stopwords = set(f.read().splitlines())
    print(f"✅ Đã tải {len(stopwords)} stopwords.")
except:
    stopwords = set()
    print("⚠️ Không tìm thấy file stopwords!")

# 3. Hàm Tiền xử lý chuẩn Kỹ sư AI
def professional_clean(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    
    # Xóa rác: URL, Email, thẻ HTML
    text = re.sub(r'http\S+|www\S+|<[^>]+>', '', text)
    
    # Bước 1: Dịch cụm từ lái/ghép trước
    for long_word, standard_word in teencode_dict.items():
        if len(long_word.split()) > 1: 
            text = text.replace(long_word, standard_word)
            
    # Bước 2: Tách từ sơ bộ để dịch các từ đơn
    words = text.split()
    words = [teencode_dict.get(w, w) for w in words]
    text = ' '.join(words)
    
    # Bước 3: Giữ lại chữ cái, số và dấu ! ?
    text = re.sub(r'[^a-zàáãạảăắằẵặẳâấầẫậẩèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ0-9\s!?]', ' ', text)
    
    # Bước 4: Tách từ chuẩn bằng PyVi
    text = ViTokenizer.tokenize(text)
    
    # Bước 5: Lọc từ dừng và Xóa rác 1 ký tự (NHƯNG BẢO TỒN DẤU CÂU CẢM XÚC)
    words = text.split()
    # Thêm điều kiện `or w in ['!', '?']` để giữ lại dấu chấm than và hỏi chấm
    clean_words = [w for w in words if w not in stopwords and (len(w) > 1 or w in ['!', '?'])]
    
    return ' '.join(clean_words).strip()

✅ Đã tải 1942 stopwords.


# Xử lý dữ liệu ViCTSD (Dữ liệu học thuật)

In [3]:
def process_victsd_file(filename):
    path = f'data/{filename}'
    if os.path.exists(path):
        df = pd.read_csv(path)
        # ViCTSD: Comment -> text, Toxicity -> label
        df = df[['Comment', 'Toxicity']].rename(columns={'Comment': 'text_raw', 'Toxicity': 'label'})
        print(f"--- Đang xử lý {filename} ---")
        df['text_cleaned'] = df['text_raw'].apply(professional_clean)
        # Loại bỏ các dòng bị rỗng sau khi làm sạch
        df = df[df['text_cleaned'].str.len() > 0]
        return df
    return None

# Thực thi xử lý 3 file gốc
df_train = process_victsd_file('ViCTSD_train.csv')
df_val = process_victsd_file('ViCTSD_valid.csv')
df_test_internal = process_victsd_file('ViCTSD_test.csv')

print(f"\n✅ Kết quả: Train({len(df_train)}), Val({len(df_val)}), Test chuẩn({len(df_test_internal)})")

--- Đang xử lý ViCTSD_train.csv ---
--- Đang xử lý ViCTSD_valid.csv ---
--- Đang xử lý ViCTSD_test.csv ---

✅ Kết quả: Train(6983), Val(1995), Test chuẩn(996)


In [4]:
import pandas as pd # Đảm bảo đã import pandas

# Nạp và xử lý lỗi định dạng CSV (THÊM ENCODING UTF-8-SIG VÀO ĐÂY)
df_yt_raw = pd.read_csv('data/youtube_label.csv', sep=';', encoding='utf-8-sig', engine='python')

# Lấy 2 cột và đổi tên
df_yt_raw = df_yt_raw[['comment', 'label']].rename(columns={'comment': 'text_raw'})

# Làm sạch nhãn
df_yt_raw['label'] = pd.to_numeric(df_yt_raw['label'], errors='coerce')
df_yt_raw = df_yt_raw.dropna(subset=['label'])
df_yt_raw['label'] = df_yt_raw['label'].astype(int)

print("--- Đang làm sạch tập YouTube thực tế ---")
# Đảm bảo hàm professional_clean đã được chạy ở cell trước đó
df_yt_raw['text_cleaned'] = df_yt_raw['text_raw'].apply(professional_clean)
df_test_youtube = df_yt_raw[df_yt_raw['text_cleaned'].str.len() > 0]

print(f"✅ Thành công! YouTube Test: {len(df_test_youtube)} dòng sạch.")

--- Đang làm sạch tập YouTube thực tế ---
✅ Thành công! YouTube Test: 957 dòng sạch.


In [5]:
os.makedirs('data/processed', exist_ok=True)

df_train.to_csv('data/processed/train_final.csv', index=False, encoding='utf-8-sig')
df_val.to_csv('data/processed/val_final.csv', index=False, encoding='utf-8-sig')
df_test_internal.to_csv('data/processed/test_standard.csv', index=False, encoding='utf-8-sig')
df_test_youtube.to_csv('data/processed/test_youtube.csv', index=False, encoding='utf-8-sig')

print("🚀 TẤT CẢ THÀNH PHẨM ĐÃ ĐƯỢC LƯU VÀO data/processed/")

🚀 TẤT CẢ THÀNH PHẨM ĐÃ ĐƯỢC LƯU VÀO data/processed/


# Kiểm tra dữ liệu sau khi làm sạch

In [ ]:
import pandas as pd

# Kiểm tra ngẫu nhiên 5 dòng của tập YouTube sau khi làm sạch
df_check = pd.read_csv('data/processed/test_youtube.csv')

print("--- KIỂM TRA NGẪU NHIÊN KẾT QUẢ LÀM SẠCH YOUTUBE ---")
# Lấy các câu có nhãn độc hại để xem Teencode đã dịch chưa
sample_toxic = df_check[df_check['label'] == 1].sample(5)

for i, row in sample_toxic.iterrows():
    print(f"Gốc: {row['text_raw']}")
    print(f"Sạch: {row['text_cleaned']}")
    print("-" * 30)

# Đo lại OOV sau khi có từ điển mới 

--- KIỂM TRA NGẪU NHIÊN KẾT QUẢ LÀM SẠCH YOUTUBE ---
Gốc: Bay hãy chạy qua bài last night của Treasure rửa tai ngay 🥰
Sạch: bay chạy last night treasure rửa tai
------------------------------
Gốc: Vl thảm họa này tận 999N view?😂
Sạch: vãi lồn thảm_họa tận 999n view ?
------------------------------
Gốc: Cảm ơn vì nghe bài này là mình hết táo bón 🎉❤❤
Sạch: cảm_ơn táo_bón
------------------------------
Gốc: Cha mẹ kiểu gì vậy chỉ co  xảy ra      ở ngupi😢a châu hèn
Sạch: cha_mẹ kiểu co xảy ngupi châu hèn
------------------------------
Gốc: bro tưởng mik hát rất hay💔💔🥀🥀💔💔
Sạch: bro tưởng mik hát
------------------------------


In [ ]:
import pandas as pd

# 1. Nạp lại tập YouTube đã xử lý
df_test_yt = pd.read_csv('data/processed/test_youtube.csv')

print("--- KIỂM TRA CHẤT LƯỢNG TIỀN XỬ LÝ ---")
print("-" * 60)

# 2. Chọn ra 5 mẫu tiêu biểu để minh họa cho báo cáo
sample_indices = [32, 14, 19, 20, 26]

for i in sample_indices:
    if i < len(df_test_yt):
        row = df_test_yt.iloc[i]
        print(f"VĂN BẢN GỐC: {row['text_raw']}")
        print(f"VĂN BẢN SẠCH: {row['text_cleaned']}")
        print(f"NHÃN GÁN: {'Độc hại' if row['label'] == 1 else 'Bình thường'}")
        print("-" * 60)

--- KIỂM TRA CHẤT LƯỢNG TIỀN XỬ LÝ ---
------------------------------------------------------------
VĂN BẢN GỐC: Ôg tóc trắng đoạn đầu hat rõ hay mà mấy ôg bà đằng sau hát nhhuuw cc t v🎉🎉🎉
VĂN BẢN SẠCH: ôg tóc trắng đoạn đầu hat mấy ôg đằng hát nhhuuw cặc tao
NHÃN GÁN: Độc hại
------------------------------------------------------------
VĂN BẢN GỐC: Neue cmt này được 30like tôi sẽ cover bài này ở quảng trường
VĂN BẢN SẠCH: neue bình_luận 30like cover quảng_trường
NHÃN GÁN: Bình thường
------------------------------------------------------------
VĂN BẢN GỐC: một mất mát nữa cho người điếc 
VĂN BẢN SẠCH: mất_mát điếc
NHÃN GÁN: Độc hại
------------------------------------------------------------
VĂN BẢN GỐC: world cup 2026 music better 😂
VĂN BẢN SẠCH: world_cup 2026 music better
NHÃN GÁN: Bình thường
------------------------------------------------------------
VĂN BẢN GỐC: 0:35 hay nhất
VĂN BẢN SẠCH: 35
NHÃN GÁN: Bình thường
------------------------------------------------------------
